# 🔀 Flat (Peer-to-Peer) Multi-Agent System

A lightweight LLM router reads the task and picks **one specialist agent** to handle it.  
All agents are peers — none has authority over the others.

```
 User Task
     │
     ▼
  [Router LLM]  ← structured output → one of: researcher | coder | writer
     │
     ▼
 [Chosen Agent] → ReAct loop (think → tool call → observe) → Answer
```

**Strengths:** Minimal overhead, very fast, easy to extend.  
**Limitations:** Only one agent runs per task — no collaboration or multi-step replanning.

---
**Install:**
```bash
uv sync --group notebooks
```
Set `OPENAI_API_KEY` in a `.env` file.

## ⚙️ Setup

In [1]:
from typing import Literal
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from tavily import TavilyClient
from typing import Any

load_dotenv()

tavily_client = TavilyClient()

# Single LLM instance shared by the router and all agents
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✅ Setup complete.")

✅ Setup complete.


## 🛠️ Tool Definitions

Each tool wraps a real-world capability. Stubs return mock strings here — swap in actual APIs or sandboxes for production.

In [2]:
@tool
def search_web(query: str) -> dict[str, Any]:
    """Search the web for information."""
    return tavily_client.search(query)

@tool
def run_python(code: str) -> str:
    """Execute Python code and return the output."""
    try:
        exec_globals = {}
        exec(code, exec_globals)
        return str(exec_globals.get("result", "No result variable defined."))
    except Exception as e:
        return f"Error executing code: {e}"

@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file and return confirmation."""
    with open(filename, "w") as f:
        f.write(content)
    return f"[Written to {filename}]"

print("✅ Tools defined.")

✅ Tools defined.


## 🤖 Agent Definitions

`create_agent` (from `langchain.agents`) builds a compiled agent runtime that can reason, call tools, and return a final response. It replaces the legacy `create_react_agent` import path.

The `system_prompt` parameter injects a **system message** that gives each agent its persona.

In [3]:
# Each agent gets its own system prompt and an exclusive tool set.
# create_agent returns a runnable agent that can call tools and emit final messages.

researcher = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt="You are a Researcher. Find and summarise information concisely.",
)

coder = create_agent(
    model=llm,
    tools=[run_python],
    system_prompt="You are a Coder. Write and execute Python code to solve problems.",
)

writer = create_agent(
    model=llm,
    tools=[write_file],
    system_prompt="You are a Writer. Produce clear, well-structured documents.",
)

AGENTS = {"researcher": researcher, "coder": coder, "writer": writer}
print("✅ Agents compiled:", list(AGENTS.keys()))

✅ Agents compiled: ['researcher', 'coder', 'writer']


## 🔀 Structured Router

`llm.with_structured_output` constrains the LLM to return a validated Pydantic object — no string parsing, no fallback guards needed. `Literal[...]` restricts the output to exactly our three agent names.

In [4]:
class RouteDecision(BaseModel):
    """The router's output: which agent should handle this task."""
    agent: Literal["researcher", "coder", "writer"]
    reason: str  # brief justification — useful for debugging / logging

# with_structured_output wraps the LLM so it always returns RouteDecision-shaped output
router_llm = llm.with_structured_output(RouteDecision)

ROUTER_SYSTEM = (
    "You are a task router. Given a task, choose the best agent:\n"
    "  - researcher: for finding or summarising information\n"
    "  - coder: for writing or running Python code\n"
    "  - writer: for producing documents, reports, or written content"
)


def run_flat_mas(task: str) -> dict[str, Any]:
    """Route the task to a single agent and return its final answer."""
    # Step 1: structured routing with explicit validation for static type checkers
    decision_raw = router_llm.invoke([
        {"role": "system", "content": ROUTER_SYSTEM},
        {"role": "user",   "content": task},
    ])
    decision = RouteDecision.model_validate(decision_raw)
    print(f"\n[Router] → agent: '{decision.agent}'  reason: {decision.reason}")

    # Step 2: run the chosen agent
    agent   = AGENTS[decision.agent]
    result  = agent.invoke({"messages": [{"role": "user", "content": task}]})

    # The final answer is the content of the last message in the state
    return result

## ▶️ Demo

The router should select **researcher** for this information-gathering task.

In [7]:
result = run_flat_mas("""Create a poem in sytle of Shakespeare about the benefits of AI agents.
                      Then store it in a file called 'poem.txt'""")
print("\n=== FLAT MAS OUTPUT ===")
print(result["messages"][-1].content)


[Router] → agent: 'writer'  reason: The task involves creating a poem in the style of Shakespeare, which is a creative writing task. The writer is best suited for producing written content, including poetry.

=== FLAT MAS OUTPUT ===
I have created a Shakespearean-style poem about the benefits of AI agents and stored it in a file named "poem.txt".


In [8]:
from pprint import pprint

pprint(result)

{'messages': [HumanMessage(content="Create a poem in sytle of Shakespeare about the benefits of AI agents.\n                      Then store it in a file called 'poem.txt'", additional_kwargs={}, response_metadata={}, id='41fee659-8e81-4052-b9c2-57b782757242'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 313, 'prompt_tokens': 92, 'total_tokens': 405, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_92d15debfd', 'id': 'chatcmpl-DPAGwq5l1STwC6UvyxZUUVhOSYKno', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3fb1-10a1-7b02-93a7-923f85d6ffc8-0', tool_calls=[{'name': 'write_file', 'args': {'filename': 'poem.txt', 'content

## 💡 Key Takeaways

- **`create_agent`** from `langchain.agents` is the current API for tool-using agents.
- **`llm.with_structured_output(Pydantic)`** replaces `prompt | llm | StrOutputParser()` + manual `.strip().lower()` for routing — outputs are always valid.
- Adding a new specialist: define its tools, call `create_agent`, add to `AGENTS`, and update `RouteDecision.agent`'s `Literal`.
- For tasks requiring **multiple sequential steps**, see `02_hierarchical_mas.ipynb`.